1. Create a simple blockchain in Python with hashing & linked blocks

In [2]:
import hashlib
import json 
import time

Steps 👍
Step 1. Create a hashing function

In [3]:
def sha256(data: str) -> str:
    """Generate SHA-256 hash of the given data."""
    return hashlib.sha256(data.encode()).hexdigest()

converts any text into a fixed 64 character string

Step 2: Create block class, include hash computing function
following attributes will be considered :


-index --> position in the chain
-timestamp --> when the block was created.
-data--> information stored 
-previous_hash --> links this block to the one before
-nonce --> a number used for mining (proof-of-work)
-hash --> unique fingerprint of the block , hash calculations

In [4]:
class Block:
    def __init__(self, index: int, timestamp: float, data: dict, previous_hash: str, nonce: int = 0):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True, separators=(",", ":"))
        return sha256(block_string)

Hash calculation


-converts block into json string.
-hashes it with SHA-256 
-ensures every block has a unique fingerprint

Step 3 : Create a blockchain class

includes creating genesis block, mining, adding a block and validating the chain

In [5]:
class Blockchain:
    def __init__(self, difficulty: int = 3):
        self.chain = []
        self.difficulty = difficulty
        self.create_genesis_block()
    def create_genesis_block(self):
        genesis = Block(index=0, timestamp=time.time(), data={"msg": "genesis block"}, previous_hash="0")
        self.mine_block(genesis)
        self.chain.append(genesis)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]
    def is_valid_proof(self, block: Block) -> bool:
        return block.hash.startswith("0" * self.difficulty)
    
    def mine_block(self, block: Block) -> Block:
        while True:
            block.hash = block.compute_hash()
            if self.is_valid_proof(block):
                return block
            block.nonce += 1
    
    def add_block(self, data: dict) -> Block:
        new_block = Block(
            index=self.last_block.index + 1,
            timestamp=time.time(),
            data=data,
            previous_hash=self.last_block.hash
        )
        self.mine_block(new_block)
        if new_block.previous_hash != self.last_block.hash or not self.is_valid_proof(new_block):
            raise ValueError("Invalid block-rejected")
        self.chain.append(new_block)
        return new_block
    
    def is_chain_valid(self) -> bool:
        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i - 1]
            if curr.previous_hash != prev.hash:
                return False
            if curr.compute_hash() != curr.hash:
                return False
        return True

In [6]:
if __name__ == "__main__":
    bc = Blockchain(difficulty=4)
    bc.add_block({"amount": 10, "from": "Alice", "to": "Bob"})
    bc.add_block({"amount": 20, "from": "Bob", "to": "Charlie"})
    for b in bc.chain:
        print(f"Index: {b.index}")
        print(f"Timestamp: {b.timestamp}")
        print(f"Data: {b.data}")
        print(f"Previous Hash: {b.previous_hash}")
        print(f"Hash: {b.hash}")
        print("-" * 60)
    print("Is blockchain valid?", bc.is_chain_valid())

Index: 0
Timestamp: 1769075311.2005148
Data: {'msg': 'genesis block'}
Previous Hash: 0
Hash: 0000fb03b542b981cb59270d6b569f070032a221497e75aa6f63d9e067f5b69e
------------------------------------------------------------
Index: 1
Timestamp: 1769075311.7728448
Data: {'amount': 10, 'from': 'Alice', 'to': 'Bob'}
Previous Hash: 0000fb03b542b981cb59270d6b569f070032a221497e75aa6f63d9e067f5b69e
Hash: 0000d21533600f4f6f6f7099714d31de15d7dd5d5953a06e7e8960747e34374d
------------------------------------------------------------
Index: 2
Timestamp: 1769075313.007264
Data: {'amount': 20, 'from': 'Bob', 'to': 'Charlie'}
Previous Hash: 0000d21533600f4f6f6f7099714d31de15d7dd5d5953a06e7e8960747e34374d
Hash: 00008300d32855eb01acbaef81f6a4ac0dd249c00b639e8a0b33dbcf6b2b9b2f
------------------------------------------------------------
Is blockchain valid? True
